# Chapter 1 &mdash; Context-Free Patterns: Nesting and Palindromes

**Concept 11 of the Chapter 1 decomposition:** *Pattern Class II -- Context-Free Patterns*

Proper nesting needs a running count with no bound &mdash; exactly what a <b>stack</b> gives you, and what finite memory cannot.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Context-Free-Patterns/Concept-Context-Free-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Properly nested parentheses have two properties:

* the same, finite-but-unbounded, number of `(` and `)`;
* sweeping left to right, the count of `(` is **at every point** at least the count
  of `)`, with equality at the end.

That second condition demands an unbounded running count. A **palindrome** is the
other archetype: it needs the front matched against the *reverse* of the back.

Both are what a last-in-first-out memory buys you.

## 2. Definitions

### A PDA for the Dyck language

Push on `(`, pop on `)`. The stack **is** the running count.
The edge label reads `input , pop ; push`.

In [ ]:
dyck = md2mc('''PDA
IF : ( , # ; (#  -> M
M  : ( , ( ; ((  -> M
M  : ) , ( ; ''  -> M
M  : '' , # ; #  -> IF
''')
print("Dyck PDA states :", sorted(dyck["Q"]))

### A stack checker in plain Python

The same algorithm, so you can see what the PDA is doing.

In [ ]:
def dyck_ok(s):
    depth = 0
    for ch in s:
        depth += 1 if ch == '(' else -1
        if depth < 0:          # a ')' arrived with nothing to match
            return False
    return depth == 0

### Palindromes: $ww^R$ versus a copy

Reversal is cheap for a stack; copying is not. Hold onto this &mdash; Concept 12 turns on it.

In [ ]:
def is_palindrome(s):
    return s == s[::-1]

## 3. Tests

The nesting checker, on the book's examples.

In [ ]:
for s in ['', '()', '(())', '(()(()))', '()()', ')(', '(()', '())(']:
    print("%-10s properly nested? %s" % (repr(s), dyck_ok(s)))
assert dyck_ok("(()(()))") and not dyck_ok(")(") and not dyck_ok("(()")

Run the **PDA** on the same strings. `explore_pda` prints every accepting run;
`STKMAX` bounds the stack during the search.

In [ ]:
explore_pda("(())", dyck, STKMAX=6)

Palindromes, and the crucial contrast.

In [ ]:
for s in ['0110', '010', '', '0101']:
    print("%-6s palindrome? %-6s  (is it w w^R? %s)"
          % (repr(s), is_palindrome(s), s == s[::-1]))
print()
print("w w^R (palindrome) : CONTEXT-FREE -- a stack returns things reversed.")
print("w w   (a copy)     : NOT context-free -- see Concept 12.")

## 4. Animation


Watch the stack grow and shrink as the PDA reads the string. The **height of the
stack is the nesting depth** &mdash; the unbounded count a DFA could not keep.


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(dyck, FuseEdges=False)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run `dyck_ok` on `'(()'` and on `'())('`. Both fail &mdash; but for *different*
   reasons. Which clause of the definition does each violate?
2. Build a PDA for balanced `[` and `]` **mixed with** `(` and `)`, where the kinds
   must match. What extra stack symbols do you need?
3. Find the midpoint of the book's long palindrome by counting `0`s between `1`s.

In [ ]:
# Your work for the exercises above.